Imputers in PySpark

![image_1780306670946.png](./image_1780306670946.png "image_1780306670946.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Imputer


# Sample data with missing values (None = null)
data = [
    (1, 25.0, 50000.0),
    (2, None, 60000.0),
    (3, 35.0, None),
    (4, None, None),
    (5, 45.0, 75000.0),
]
df = spark.createDataFrame(data, ["id", "age", "salary"])
df.show()

In [0]:
# ── Step 1: Define the Imputer (Estimator) ──────────────────────────────────
imputer = Imputer(
    inputCols=["age", "salary"],
    outputCols=["age_imp", "salary_imp"],
    strategy="median"          # options: "mean" | "median" | "mode"
)

# ── Step 2: Fit — learns mean/median from non-null values ───────────────────
imputer_model = imputer.fit(df)

# Inspect learned values
print("Surrogate values learned:")
print(imputer_model.surrogateDF.show())

# ── Step 3: Transform — fills nulls with learned statistics ─────────────────
result = imputer_model.transform(df)
result.show()

Use Imputer inside a pipeline

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

imputer = Imputer(
    inputCols=["age", "salary"],
    outputCols=["age_imp", "salary_imp"],
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=["age_imp", "salary_imp"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[imputer, assembler, lr])

# .fit() trains ALL stages sequentially (including Imputer)
pipeline_model = pipeline.fit(train_df)
predictions = pipeline_model.transform(test_df)